# WasteWise AI — CNN Training Walkthrough

This notebook trains a five-class recyclable-waste image classifier.

**Classes:** cardboard, glass, metal, paper, plastic

Run `python training/prepare_dataset.py` before using this notebook.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

DATA_DIR = Path("../data/processed")
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## Load the three dataset partitions

Training data updates the model weights. Validation data guides model selection.
Test data is reserved for the final unbiased evaluation.

In [ ]:
common = {
    "image_size": (IMAGE_SIZE, IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "label_mode": "int",
}

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", shuffle=True, seed=SEED, **common
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "val", shuffle=False, **common
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", shuffle=False, **common
)

class_names = train_ds.class_names
class_names

## Visual sanity check

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for index in range(min(12, len(images))):
        axis = plt.subplot(3, 4, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(class_names[int(labels[index])])
        plt.axis("off")
plt.tight_layout()

## Improve input-pipeline performance

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## Build the CNN

Data augmentation creates modified training examples. Convolution layers learn
visual patterns, pooling reduces spatial size, dropout reduces overfitting, and
softmax produces a probability distribution over the five classes.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    augmentation,
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Train with checkpointing and early stopping

In [ ]:
checkpoint_path = ARTIFACTS_DIR / "best_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

with (ARTIFACTS_DIR / "labels.json").open("w") as file:
    json.dump(class_names, file, indent=2)

## Plot accuracy and loss

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(ARTIFACTS_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(history_df["accuracy"], label="Training")
axes[0].plot(history_df["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history_df["loss"], label="Training")
axes[1].plot(history_df["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "training_curves.png", dpi=180)
plt.show()

## Final test evaluation

In [ ]:
best_model = tf.keras.models.load_model(checkpoint_path)
test_loss, test_accuracy = best_model.evaluate(test_ds)

true_labels = []
predicted_labels = []

for images, labels in test_ds:
    probabilities = best_model.predict(images, verbose=0)
    true_labels.extend(labels.numpy().tolist())
    predicted_labels.extend(np.argmax(probabilities, axis=1).tolist())

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv(ARTIFACTS_DIR / "classification_report.csv")

matrix = confusion_matrix(true_labels, predicted_labels)
display_plot = ConfusionMatrixDisplay(matrix, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
display_plot.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("WasteWise Test Confusion Matrix")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "confusion_matrix.png", dpi=180)
plt.show()

metrics = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(report["macro avg"]["precision"]),
    "macro_recall": float(report["macro avg"]["recall"]),
    "macro_f1": float(report["macro avg"]["f1-score"]),
}
with (ARTIFACTS_DIR / "metrics.json").open("w") as file:
    json.dump(metrics, file, indent=2)

metrics

## Interpretation checklist

Record:

1. Best validation accuracy
2. Final test accuracy
3. Macro F1-score
4. Most confused class pair
5. Whether training and validation curves separate
6. Results on photographs not taken from the dataset
7. Examples rejected by the confidence threshold